# MelBERT — Kaggle reproduction notebook

Notebook chạy lại **MelBERT** bằng code chính thức [`jin530/MelBERT`](https://github.com/jin530/MelBERT), với các thiết lập trong paper *MelBERT: Metaphor Detection via Contextualized Late Interaction using Metaphorical Identification Theories* (NAACL-HLT 2021).

## Protocol được triển khai

| Dataset | Vai trò | Cách chạy |
|---|---|---|
| VUA-18 | Train + test | `train.tsv`; theo dõi F1 qua `test.tsv` như `main.py` của repo |
| VUA-20 | Train + test | `train.tsv`; theo dõi F1 qua `test.tsv` như `main.py` của repo |
| MOH-X | Zero-shot test | Không train; dùng model đã train trên VUA-20; đánh giá 10 split |
| TroFi | Zero-shot test | Không train; dùng model đã train trên VUA-20; đánh giá 10 split |

**Không train riêng trên MOH-X/TroFi.** Paper dùng các tập này để đo khả năng chuyển miền zero-shot của model được train trên VUA-20.

## Cấu hình

`roberta-base`, `MELBERT`, `classifier_hidden=768`, `dropout=0.2`, `use_pos=True`, `use_local_context=True`, `max_seq_length=150`, `train_batch_size=32`, `eval_batch_size=8`, `learning_rate=3e-5`, `warmup_epoch=2`, `num_train_epoch=3`, `class_weight=3`, AdamW, weight decay `0.01`, gradient clipping `1.0`.

Paper nói kết quả được lấy trung bình trên 5 random seeds nhưng không ghi rõ các seed trong phần implementation details. Notebook mặc định dùng `[1, 2, 3, 4, 5]`, theo hướng dẫn reproduction bổ sung của repository [ms_wilde](https://github.com/languagemedialaboratory/ms_wilde), vốn sử dụng code MelBERT gốc.

## Kaggle setup

1. Bật **GPU** và **Internet** trong Notebook settings.
2. Chạy các cell từ trên xuống. Dataset gốc được tải từ Google Drive link công khai đặt trong README repo.
3. Chạy lại notebook sau khi timeout để tiếp tục từ checkpoint. Để mang checkpoint qua một kernel/session mới, hãy **Save Version** khi notebook dừng và attach output của version đó vào notebook tiếp theo (Add Input); notebook sẽ tự dò `artifact_manifest.json` trong `/kaggle/input` và copy checkpoint về `/kaggle/working`.

> Kaggle không đảm bảo `/kaggle/working` tồn tại sau khi kernel bị reset. Checkpoint được ghi thường xuyên và xuất hiện trong Notebook Output khi version kết thúc; cần attach output cũ làm input cho session mới. Notebook chủ động dừng trước giới hạn 12 giờ để có thời gian ghi output. Không có cách nào bảo đảm lưu được file local nếu tiến trình bị kill đột ngột trước khi Kaggle lưu output.

## Lưu ý tái lập

- Repo gốc ghi `python==3.7`, `torch==1.6.0`, `transformers==4.2.2`; đây là stack cũ, thường không cài trực tiếp được trên runtime Kaggle/Python hiện tại. Notebook giữ nguyên model/data pipeline và hyperparameters, dùng Transformers 4.57.1 + tokenizers 0.22.2 + huggingface-hub 0.36.0 (wheel tương thích Python 3.13; pin hub <1.0 để tránh xung đột với hub 1.33.0), đồng thời vá import `AdamW` sang `torch.optim.AdamW`. Epsilon được đặt tường minh `1e-6` để khớp mặc định của AdamW trong Transformers 4.2.2. Paper gọi optimizer là Adam ở phần mô tả; source repo thực tế dùng AdamW và notebook ưu tiên source chính thức.
- `main.py` gốc đánh giá `test.tsv` sau mỗi epoch trong `run_train` và lưu model mỗi epoch cho task `vua`. Notebook giữ hành vi đó để bám sát source code. Điều này có nghĩa test F1 được quan sát qua epoch; cần ghi rõ caveat này khi báo cáo reproduction.
- Trong source hiện tại, `run_train()` gọi `model.train()` một lần trước vòng epoch, còn `run_eval()` gọi `model.eval()` và không khôi phục train mode. Vì vậy epoch 2–3 tiếp tục chạy ở eval mode. Notebook giữ nguyên behavior này để reproduction đúng repo (không tự ý “sửa” bug); checkpoint lưu cờ `model.training` để resume không đổi behavior.
- Kaggle GPU có thể khác TITAN RTX trong paper; vì thế kết quả có thể dao động dù cấu hình dữ liệu/mô hình giống nhau.


In [1]:
# Record the start time early: Kaggle's 12-hour limit includes setup/download time.
import os, sys, time, json, shutil, subprocess, zipfile, csv, random, math, re, gc
from pathlib import Path
from datetime import datetime, timezone

NOTEBOOK_START_TIME = time.monotonic()
MAX_SESSION_HOURS = 10.75       # stop before Kaggle's nominal 12-hour hard limit
CHECKPOINT_EVERY_STEPS = 250  # full model + Adam states are large; safe-stop still forces an immediate save
STOP_RESERVE_MINUTES = 35

print("Python:", sys.version)
print("Notebook start (UTC):", datetime.now(timezone.utc).isoformat())


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Notebook start (UTC): 2026-11-26T07:05:43.727509+00:00


In [2]:
# Kaggle Python 3.13 / PyTorch 2.11 compatibility stack.
# Pin the FULL Hugging Face dependency pair: Transformers 4.57.1 requires
# huggingface-hub >=0.34,<1.0 (Kaggle may ship hub 1.33.0, which is incompatible).
# Use wheels only for tokenizers; never compile Rust from source.
%pip -q install "setuptools==80.9.0"
%pip -q install --only-binary=:all: "tokenizers==0.22.2"
%pip -q install "huggingface-hub==0.36.0"
%pip -q install --no-deps "transformers==4.57.1"
%pip -q install "gdown==5.2.0"

import importlib.metadata as _md, sys
print("Python:", sys.version.split()[0])
for _pkg in ("setuptools", "tokenizers", "huggingface-hub", "transformers", "gdown"):
    print(f"{_pkg}: {_md.version(_pkg)}")

# Required compatibility checks before importing Transformers.
from packaging.version import Version
assert Version(_md.version("huggingface-hub")) >= Version("0.34.0")
assert Version(_md.version("huggingface-hub")) < Version("1.0")
assert Version(_md.version("tokenizers")) >= Version("0.22.0")
assert Version(_md.version("tokenizers")) <= Version("0.23.0")
print("Dependencies installed. Restart the Kaggle session once, then Run All from the top.")


Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.20.2 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.20.2 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.0 which is incompatible.
transformers 5.20.2 requires tokenizers<0.24.0,>=0.23.1, but you have tokenizers 0.22.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.0 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you

In [3]:
import torch
import numpy as np
import tokenizers
import huggingface_hub
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

if transformers.__version__ != "4.57.1":
    raise RuntimeError(f"Expected transformers 4.57.1, got {transformers.__version__}")
if tokenizers.__version__ != "0.22.2":
    raise RuntimeError(f"Expected tokenizers 0.22.2, got {tokenizers.__version__}")
if not (huggingface_hub.__version__.startswith("0.") and
        tuple(map(int, huggingface_hub.__version__.split(".")[:2])) >= (0, 34)):
    raise RuntimeError(
        f"Expected huggingface-hub >=0.34,<1.0, got {huggingface_hub.__version__}. "
        "Rerun the dependency cell, restart the session, then Run All."
    )
if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa bật. Vào Notebook Settings → Accelerator → GPU rồi chạy lại.")
print("GPU:", torch.cuda.get_device_name(0))

# Match the older numerical path as closely as modern PyTorch allows.
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "matmul"):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.allow_tf32 = False


torch: 2.11.0+cu130
transformers: 4.57.1
tokenizers: 0.22.2
huggingface_hub: 0.36.0
CUDA available: True
GPU: Tesla T4


In [4]:
# Clone official source. The original repository is used for model architecture,
# feature conversion, processors, evaluation and checkpoint serialization.
REPO_URL = "https://github.com/jin530/MelBERT.git"
REPO_DIR = Path("/kaggle/working/MelBERT")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    print("Using existing repo:", REPO_DIR)

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print("MelBERT commit:", commit)

# Minimal runtime-compatibility patch: Transformers' AdamW export is deprecated/removed
# in some newer versions; torch.optim.AdamW implements the same decoupled AdamW update.
main_py = REPO_DIR / "main.py"
source = main_py.read_text(encoding="utf-8")
old_import = "from transformers import AutoTokenizer, AutoModel, AdamW, get_linear_schedule_with_warmup"
new_import = "from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup\nfrom torch.optim import AdamW"
if old_import in source:
    source = source.replace(old_import, new_import)
    main_py.write_text(source, encoding="utf-8")
elif "from torch.optim import AdamW" not in source:
    raise RuntimeError("Không tìm thấy import line mong đợi trong main.py; repo có thể đã thay đổi.")

sys.path.insert(0, str(REPO_DIR))
import main as melbert_main
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import NLLLoss

print("Loaded official repo modules:", melbert_main.__file__)


Cloning into '/kaggle/working/MelBERT'...


MelBERT commit: 11b0f78dae93c71c56b805dc28d810f6050c86c4
Loaded official repo modules: /kaggle/working/MelBERT/main.py


## Tải và định vị bốn bộ dữ liệu

README của repo cung cấp một Google Drive archive chứa các TSV đã được chuẩn hóa theo định dạng repo. Cell bên dưới tải archive và dò các thư mục `VUA18`, `VUA20`, `MOH-X/CLS`, `TroFi/CLS` theo tên thực tế sau giải nén. Có thể thay `DATA_ARCHIVE_ID` nếu tác giả đổi file chia sẻ.


In [5]:
DATA_ARCHIVE_ID = "18hemekvUuOw-qkQhWv4F6qFnis6lkaog"  # naacl2021_melbert_data.zip linked by official README
DATA_ROOT = Path("/kaggle/working/melbert_data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ARCHIVE_PATH = Path("/kaggle/working/melbert_data.zip")

# If you already attached a Kaggle Dataset containing the official MelBERT TSV folders,
# set MANUAL_DATA_ROOT to that path (e.g. /kaggle/input/my-melbert-data).
MANUAL_DATA_ROOT = ""

if MANUAL_DATA_ROOT:
    DATA_SEARCH_ROOT = Path(MANUAL_DATA_ROOT)
else:
    if not ARCHIVE_PATH.exists() or ARCHIVE_PATH.stat().st_size < 1024:
        import gdown
        downloaded = gdown.download(id=DATA_ARCHIVE_ID, output=str(ARCHIVE_PATH), quiet=False)
        if not downloaded:
            raise RuntimeError("Google Drive download thất bại. Bật Internet hoặc dùng MANUAL_DATA_ROOT trỏ tới dataset đã attach.")
    if not zipfile.is_zipfile(ARCHIVE_PATH):
        raise RuntimeError("File tải về không phải ZIP. Kiểm tra quyền chia sẻ Drive hoặc thay nguồn dataset.")
    marker = DATA_ROOT / ".extracted_ok"
    if not marker.exists():
        with zipfile.ZipFile(ARCHIVE_PATH) as zf:
            # Reject path traversal entries before extracting an externally hosted archive.
            root_resolved = DATA_ROOT.resolve()
            for member in zf.infolist():
                target = (DATA_ROOT / member.filename).resolve()
                if root_resolved not in target.parents and target != root_resolved:
                    raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
            zf.extractall(DATA_ROOT)
        marker.write_text("ok", encoding="utf-8")
    DATA_SEARCH_ROOT = DATA_ROOT

print("Dataset search root:", DATA_SEARCH_ROOT)


Downloading...
From: https://drive.google.com/uc?id=18hemekvUuOw-qkQhWv4F6qFnis6lkaog
To: /kaggle/working/melbert_data.zip
100%|██████████| 16.3M/16.3M [00:00<00:00, 72.5MB/s]


Dataset search root: /kaggle/working/melbert_data


In [6]:
def norm_path(p):
    return re.sub(r"[^a-z0-9]", "", str(p).lower())

def count_tsv_rows(path):
    with open(path, "r", encoding="utf-8", newline="") as f:
        return max(sum(1 for _ in csv.reader(f, delimiter="\t")) - 1, 0)  # exclude header

def find_vua_dir(root: Path, tag: str):
    candidates = []
    for train_file in root.rglob("train.tsv"):
        d = train_file.parent
        if (d / "test.tsv").exists() and tag in norm_path(d):
            candidates.append(d)
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy thư mục {tag} có train.tsv + test.tsv dưới {root}")
    # Prefer the most specific/deepest directory, not an outer parent.
    return sorted(candidates, key=lambda p: len(p.parts), reverse=True)[0]

def find_fold_dir(root: Path, tag: str):
    candidates = []
    for test0 in root.rglob("test0.tsv"):
        d = test0.parent
        if tag in norm_path(d) and all((d / f"test{k}.tsv").exists() for k in range(10)):
            candidates.append(d)
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy {tag} với test0.tsv ... test9.tsv dưới {root}")
    return sorted(candidates, key=lambda p: len(p.parts), reverse=True)[0]

VUA18_DIR = find_vua_dir(DATA_SEARCH_ROOT, "vua18")
VUA20_DIR = find_vua_dir(DATA_SEARCH_ROOT, "vua20")
MOHX_DIR = find_fold_dir(DATA_SEARCH_ROOT, "mohx")
TROFI_DIR = find_fold_dir(DATA_SEARCH_ROOT, "trofi")

DATA_DIRS = {
    "VUA18": VUA18_DIR,
    "VUA20": VUA20_DIR,
    "MOH-X": MOHX_DIR,
    "TroFi": TROFI_DIR,
}
for name, directory in DATA_DIRS.items():
    print(f"{name:7s} -> {directory}")


VUA18   -> /kaggle/working/melbert_data/VUA18
VUA20   -> /kaggle/working/melbert_data/VUA20
MOH-X   -> /kaggle/working/melbert_data/MOH-X/CLS
TroFi   -> /kaggle/working/melbert_data/TroFi/CLS


## Reproduction configuration and checkpoint storage

The model and data conversion functions are imported from the cloned repository. Checkpoints store **model + AdamW + scheduler + epoch/batch cursor + exact current-epoch sample order + Python/NumPy/Torch/CUDA RNG states**. This is necessary to resume in the middle of an epoch rather than restarting that epoch with a different shuffled order.

Checkpoint file is written atomically (`.tmp` then rename) to avoid replacing a valid checkpoint with a partially written one.


In [7]:
# Experimental settings (paper + official main_config.cfg)
SEEDS = [1, 2, 3, 4, 5]
BASE_MODEL = "roberta-base"
MODEL_TYPE = "MELBERT"
CLASSIFIER_HIDDEN = 768
DROPOUT = 0.2
USE_POS = True
USE_LOCAL_CONTEXT = True
MAX_SEQ_LENGTH = 150
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 3e-5
WARMUP_EPOCHS = 2
NUM_EPOCHS = 3
CLASS_WEIGHT = 3.0
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
KFOLD = 10

ARTIFACT_DIR = Path("/kaggle/working/melbert_reproduction")
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
MODEL_DIR = ARTIFACT_DIR / "models"
RESULTS_DIR = ARTIFACT_DIR / "results"
for p in [ARTIFACT_DIR, CHECKPOINT_DIR, MODEL_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Restore files from an attached output of an earlier notebook version, if present.
manifest_path = ARTIFACT_DIR / "artifact_manifest.json"
if not manifest_path.exists():
    previous_manifests = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for candidate in kaggle_input.rglob("artifact_manifest.json"):
            try:
                info = json.loads(candidate.read_text(encoding="utf-8"))
                if info.get("project") == "MelBERT-Kaggle-Reproduction":
                    previous_manifests.append(candidate)
            except Exception:
                pass
    if previous_manifests:
        previous_manifests.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        source_dir = previous_manifests[0].parent
        shutil.copytree(source_dir, ARTIFACT_DIR, dirs_exist_ok=True)
        print("Restored previous run artifacts from:", source_dir)
    else:
        print("No previous artifact_manifest.json found in /kaggle/input — starting a fresh run.")

manifest = {
    "project": "MelBERT-Kaggle-Reproduction",
    "repo_url": REPO_URL,
    "repo_commit": commit,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seeds": SEEDS,
    "config": {
        "model": BASE_MODEL, "model_type": MODEL_TYPE, "classifier_hidden": CLASSIFIER_HIDDEN,
        "dropout": DROPOUT, "use_pos": USE_POS, "use_local_context": USE_LOCAL_CONTEXT,
        "max_seq_length": MAX_SEQ_LENGTH, "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "warmup_epochs": WARMUP_EPOCHS, "epochs": NUM_EPOCHS,
        "class_weight": CLASS_WEIGHT, "weight_decay": WEIGHT_DECAY,
        "max_grad_norm": MAX_GRAD_NORM, "kfold": KFOLD,
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

CURRENT_CHECKPOINT = CHECKPOINT_DIR / "current_training.pt"
PROGRESS_PATH = RESULTS_DIR / "progress.json"

def read_json(path, default):
    if Path(path).exists():
        return json.loads(Path(path).read_text(encoding="utf-8"))
    return default

def write_json_atomic(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)

progress = read_json(PROGRESS_PATH, {
    "completed_training": {},
    "transfer_results": {},
    "transfer_completed": [],
})

print("Artifact directory:", ARTIFACT_DIR)
print("Seeds:", SEEDS)
print("Checkpoint interval:", CHECKPOINT_EVERY_STEPS, "optimizer steps")


Restored previous run artifacts from: /kaggle/input/notebooks/gre4te4r/melbert/melbert_reproduction
Artifact directory: /kaggle/working/melbert_reproduction
Seeds: [1, 2, 3, 4, 5]
Checkpoint interval: 250 optimizer steps


In [8]:
class NotebookLogger:
    def info(self, message):
        print(message)
    def warning(self, message):
        print("WARNING:", message)
    def debug(self, message):
        pass

logger = NotebookLogger()

def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def build_args(seed: int, task_name: str, data_dir: Path, log_dir: Path):
    from types import SimpleNamespace
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return SimpleNamespace(
        bert_model=BASE_MODEL,
        data_dir=str(data_dir),
        task_name=task_name,
        model_type=MODEL_TYPE,
        classifier_hidden=CLASSIFIER_HIDDEN,
        lr_schedule="warmup_linear",
        warmup_epoch=WARMUP_EPOCHS,
        drop_ratio=DROPOUT,
        kfold=KFOLD,
        num_bagging=0,
        bagging_index=0,
        use_pos=USE_POS,
        use_local_context=USE_LOCAL_CONTEXT,
        max_seq_length=MAX_SEQ_LENGTH,
        do_train=True,
        do_test=True,
        do_eval=True,
        do_lower_case=False,
        class_weight=CLASS_WEIGHT,
        train_batch_size=TRAIN_BATCH_SIZE,
        eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        num_train_epoch=NUM_EPOCHS,
        no_cuda=False,
        seed=seed,
        num_labels=2,
        n_gpu=torch.cuda.device_count(),
        device=device,
        log_dir=str(log_dir),
    )

def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }

def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch_cpu"])
    if torch.cuda.is_available() and state.get("torch_cuda") is not None:
        torch.cuda.set_rng_state_all(state["torch_cuda"])

def torch_load_full(path, map_location="cpu"):
    # PyTorch 2.6+ defaults to weights_only=True; RNG/optimizer checkpoints need full pickle state.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def save_training_checkpoint(path, payload):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, path)

def remaining_session_seconds():
    return MAX_SESSION_HOURS * 3600 - (time.monotonic() - NOTEBOOK_START_TIME)

print("Remaining execution budget (hours):", round(remaining_session_seconds() / 3600, 2))


Remaining execution budget (hours): 10.73


## Training runner có thể resume giữa epoch

`load_train_data`, `load_test_data`, `load_pretrained_model`, `run_eval` và `save_model` đều đến từ repo chính thức. Vòng lặp train dưới đây bám logic `run_train` trong `main.py`, bổ sung lưu checkpoint theo batch. Thứ tự mẫu của epoch hiện tại được lưu để khi resume không tạo permutation mới.


In [9]:
def build_optimizer_and_scheduler(model, train_loader):
    no_decay = ["bias", "LayerNorm.bias", "LayerNorm.weight"]
    named = list(model.named_parameters())
    grouped = [
        {"params": [p for n, p in named if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in named if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = AdamW(grouped, lr=LEARNING_RATE, eps=1e-6)  # HF Transformers 4.2.2 AdamW default epsilon
    total_steps = len(train_loader) * NUM_EPOCHS
    scheduler = transformers.get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_EPOCHS * len(train_loader)),
        num_training_steps=total_steps,
    )
    return optimizer, scheduler

def forward_loss(batch, model, args):
    batch = tuple(t.to(args.device) for t in batch)
    (input_ids, input_mask, segment_ids, label_ids,
     input_ids_2, input_mask_2, segment_ids_2) = batch
    logits = model(
        input_ids,
        input_ids_2,
        target_mask=(segment_ids == 1),
        target_mask_2=segment_ids_2,
        attention_mask_2=input_mask_2,
        token_type_ids=segment_ids,
        attention_mask=input_mask,
    )
    loss_fct = NLLLoss(weight=torch.tensor([1.0, args.class_weight], device=args.device))
    return loss_fct(logits.view(-1, args.num_labels), label_ids.view(-1))

def train_one_job(dataset_name: str, seed: int):
    """Train one VUA run; returns True if complete, False if safely paused."""
    assert dataset_name in ("VUA18", "VUA20")
    job_id = f"{dataset_name}_seed{seed}"
    if job_id in progress["completed_training"]:
        print("Already completed:", job_id)
        return True

    data_dir = DATA_DIRS[dataset_name]
    final_dir = MODEL_DIR / dataset_name / f"seed_{seed}"
    final_dir.mkdir(parents=True, exist_ok=True)
    args = build_args(seed, "vua", data_dir, final_dir)

    # Seed before model construction, matching original main.py initialization order.
    set_all_seeds(seed)
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, do_lower_case=False)
    processor = melbert_main.processors["vua"]()
    label_list = processor.get_labels()
    output_mode = melbert_main.output_modes["vua"]
    train_loader = melbert_main.load_train_data(
        args, logger, processor, "vua", label_list, tokenizer, output_mode
    )

    model = melbert_main.load_pretrained_model(args)
    optimizer, scheduler = build_optimizer_and_scheduler(model, train_loader)

    start_epoch = 0
    next_batch = 0
    epoch_order = None
    global_step = 0
    best_f1 = -1.0
    best_result = {}

    restored_current_job = False
    if CURRENT_CHECKPOINT.exists():
        saved = torch_load_full(CURRENT_CHECKPOINT, map_location="cpu")
        if saved.get("job_id") == job_id:
            restored_current_job = True
            if saved.get("repo_commit") != commit:
                raise RuntimeError("Repo commit khác checkpoint hiện tại; không thể resume an toàn.")
            model.load_state_dict(saved["model_state"])
            optimizer.load_state_dict(saved["optimizer_state"])
            scheduler.load_state_dict(saved["scheduler_state"])
            start_epoch = int(saved["epoch"])
            next_batch = int(saved["next_batch"])
            epoch_order = saved["epoch_order"]
            global_step = int(saved["global_step"])
            best_f1 = float(saved["best_f1"])
            best_result = saved["best_result"]
            restore_rng_state(saved["rng_state"])
            model.train(bool(saved.get("model_training", True)))
            print(f"RESUME {job_id}: epoch={start_epoch+1}, next batch={next_batch}, global_step={global_step}, model.training={model.training}")
        else:
            print("Removing stale checkpoint for another job:", saved.get("job_id"))
            CURRENT_CHECKPOINT.unlink()

    def persist(epoch, batch_cursor, order):
        payload = {
            "job_id": job_id,
            "repo_commit": commit,
            "dataset_name": dataset_name,
            "seed": seed,
            "epoch": epoch,
            "next_batch": batch_cursor,
            "epoch_order": order,
            "global_step": global_step,
            "best_f1": best_f1,
            "best_result": best_result,
            "model_state": model.state_dict(),
            "model_training": bool(model.training),
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "rng_state": capture_rng_state(),
            "saved_utc": datetime.now(timezone.utc).isoformat(),
        }
        save_training_checkpoint(CURRENT_CHECKPOINT, payload)

    # Original main.py calls model.train() once before its epoch loop. run_eval() later
    # switches it to eval mode and the original code does not switch it back in later epochs.
    # On a fresh job this is already the default; on resume restore the saved mode above.
    if not restored_current_job:
        model.train()
    print(f"\n===== START {job_id} =====")
    print("Train examples:", len(train_loader.dataset), "batches/epoch:", len(train_loader))

    for epoch in range(start_epoch, NUM_EPOCHS):
        if epoch_order is None:
            # Original DataLoader iterator consumes one global RNG draw for its base seed;
            # RandomSampler then draws its own seed/permutation. Reproduce that sequence.
            _ = torch.empty((), dtype=torch.int64).random_().item()
            epoch_order = list(iter(train_loader.sampler))
            next_batch = 0

        start_sample = next_batch * TRAIN_BATCH_SIZE
        remaining_indices = epoch_order[start_sample:]
        # Use a private DataLoader generator: creating this resume iterator must not consume
        # the global torch RNG saved for model dropout and subsequent operations.
        resume_generator = torch.Generator()
        resume_generator.manual_seed(123456)
        resume_loader = DataLoader(
            train_loader.dataset,
            sampler=remaining_indices,
            batch_size=TRAIN_BATCH_SIZE,
            drop_last=False,
            num_workers=0,
            generator=resume_generator,
        )
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} — resuming at batch {next_batch}/{len(train_loader)}")

        for batch in resume_loader:
            loss = forward_loss(batch, model, args)
            if args.n_gpu > 1:
                loss = loss.mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            global_step += 1
            next_batch += 1

            # Save at a bounded interval, and force a save once the safe-stop threshold is reached.
            reserve_seconds = STOP_RESERVE_MINUTES * 60
            force_stop = remaining_session_seconds() <= reserve_seconds
            if (global_step % CHECKPOINT_EVERY_STEPS == 0) or force_stop:
                persist(epoch, next_batch, epoch_order)
                print(f"Checkpoint saved: step={global_step}, epoch={epoch+1}, next_batch={next_batch}")
                if force_stop:
                    print("Safe stop: remaining Kaggle runtime is below reserve threshold.")
                    return False

        # Same evaluation call used by main.py's run_train: evaluate the dataset's test.tsv each epoch.
        all_guids, eval_loader = melbert_main.load_test_data(
            args, logger, processor, "vua", label_list, tokenizer, output_mode
        )
        result = melbert_main.run_eval(args, logger, model, eval_loader, all_guids, "vua")
        if result["f1"] > best_f1:
            best_f1 = float(result["f1"])
            best_result = {k: float(v) for k, v in result.items()}

        # Official source saves the VUA model every epoch (therefore the file after epoch 3 is final epoch).
        melbert_main.save_model(args, model, tokenizer)

        # Epoch-boundary checkpoint: next epoch starts with a fresh official RandomSampler permutation.
        start_epoch = epoch + 1
        next_batch = 0
        epoch_order = None
        persist(start_epoch, next_batch, epoch_order)
        print(f"Epoch {epoch+1} finished. metrics={result}; best_so_far={best_result}")

        if remaining_session_seconds() <= STOP_RESERVE_MINUTES * 60:
            print("Safe stop after epoch boundary.")
            return False

    # Record the completed seed; save_model() above wrote the official pytorch_model.bin and tokenizer files.
    entry = {
        "dataset": dataset_name,
        "seed": seed,
        "best_test_metrics_over_epochs": best_result,
        "final_model_dir": str(final_dir),
        "final_model_path": str(final_dir / "pytorch_model.bin"),
        "repo_commit": commit,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    progress["completed_training"][job_id] = entry
    write_json_atomic(PROGRESS_PATH, progress)
    if CURRENT_CHECKPOINT.exists():
        CURRENT_CHECKPOINT.unlink()
    print("COMPLETED:", entry)
    del model, optimizer, scheduler, train_loader
    gc.collect()
    torch.cuda.empty_cache()
    return True


## Chạy 10 training jobs

Mỗi dataset VUA chạy 5 seeds, mỗi seed 3 epochs. Nếu còn ít hơn `STOP_RESERVE_MINUTES` trước ngưỡng an toàn, cell sẽ dừng sau khi ghi checkpoint; chạy lại notebook sẽ tiếp tục job dang dở và bỏ qua các job đã hoàn thành.


In [10]:
TRAINING_PLAN = [(name, seed) for name in ["VUA18", "VUA20"] for seed in SEEDS]

for dataset_name, seed in TRAINING_PLAN:
    job_id = f"{dataset_name}_seed{seed}"
    if job_id in progress["completed_training"]:
        print("SKIP completed:", job_id)
        continue
    if remaining_session_seconds() <= STOP_RESERVE_MINUTES * 60:
        print("Not enough safe runtime left; stopping before starting", job_id)
        break
    completed = train_one_job(dataset_name, seed)
    if not completed:
        print("Training paused safely. Attach this notebook version's output to the next session, then rerun.")
        break

print("Training jobs completed:", len(progress["completed_training"]), "/", len(TRAINING_PLAN))
write_json_atomic(PROGRESS_PATH, progress)


SKIP completed: VUA18_seed1
SKIP completed: VUA18_seed2
SKIP completed: VUA18_seed3
SKIP completed: VUA18_seed4
SKIP completed: VUA18_seed5
SKIP completed: VUA20_seed1
SKIP completed: VUA20_seed2
SKIP completed: VUA20_seed3


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (10 outputs left out)

Evaluating: 100%|██████████| 2775/2775 [05:04<00:00,  9.10it/s]


  acc = 0.8943953865561363
  f1 = 0.6585081585081585
  precision = 0.7841776544066621
  recall = 0.5675539929683576
Epoch 2 finished. metrics={'acc': np.float64(0.8943953865561363), 'precision': 0.7841776544066621, 'recall': 0.5675539929683576, 'f1': 0.6585081585081585}; best_so_far={'acc': 0.8943953865561363, 'precision': 0.7841776544066621, 'recall': 0.5675539929683576, 'f1': 0.6585081585081585}
Epoch 3/3 — resuming at batch 0/5005
Checkpoint saved: step=10250, epoch=3, next_batch=240
Checkpoint saved: step=10500, epoch=3, next_batch=490
Checkpoint saved: step=10750, epoch=3, next_batch=740
Checkpoint saved: step=11000, epoch=3, next_batch=990
Checkpoint saved: step=11250, epoch=3, next_batch=1240
Checkpoint saved: step=11500, epoch=3, next_batch=1490
Checkpoint saved: step=11750, epoch=3, next_batch=1740
Checkpoint saved: step=12000, epoch=3, next_batch=1990
Checkpoint saved: step=12250, epoch=3, next_batch=2240
Checkpoint saved: step=12500, epoch=3, next_batch=2490
Checkpoint saved

Evaluating: 100%|██████████| 2775/2775 [05:04<00:00,  9.11it/s]


  acc = 0.9041268697062533
  f1 = 0.7261261261261261
  precision = 0.7447201689545935
  recall = 0.7084379708689101
Epoch 3 finished. metrics={'acc': np.float64(0.9041268697062533), 'precision': 0.7447201689545935, 'recall': 0.7084379708689101, 'f1': 0.7261261261261261}; best_so_far={'acc': 0.9041268697062533, 'precision': 0.7447201689545935, 'recall': 0.7084379708689101, 'f1': 0.7261261261261261}
COMPLETED: {'dataset': 'VUA20', 'seed': 5, 'best_test_metrics_over_epochs': {'acc': 0.9041268697062533, 'precision': 0.7447201689545935, 'recall': 0.7084379708689101, 'f1': 0.7261261261261261}, 'final_model_dir': '/kaggle/working/melbert_reproduction/models/VUA20/seed_5', 'final_model_path': '/kaggle/working/melbert_reproduction/models/VUA20/seed_5/pytorch_model.bin', 'repo_commit': '11b0f78dae93c71c56b805dc28d810f6050c86c4', 'completed_utc': '2026-11-26T12:50:38.152862+00:00'}
Training jobs completed: 10 / 10


## Zero-shot evaluation trên MOH-X và TroFi

Sau khi có 5 model VUA-20, chúng được nạp lại theo đúng kiến trúc MelBERT và đánh giá trên 10 test folds của từng dataset. Không cập nhật gradient, không fine-tune trên MOH-X/TroFi. Mỗi fold được ghi vào `progress.json` để nếu phiên Kaggle dừng giữa evaluation thì không cần chạy lại fold đã xong.


In [11]:
def evaluate_transfer_fold(model_path: Path, dataset_name: str, seed: int, fold: int):
    data_dir = DATA_DIRS[dataset_name]
    model_dir = MODEL_DIR / "transfer_eval" / dataset_name / f"seed_{seed}"
    model_dir.mkdir(parents=True, exist_ok=True)
    args = build_args(seed, "trofi", data_dir, model_dir)
    set_all_seeds(seed)

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, do_lower_case=False)
    processor = melbert_main.processors["trofi"]()
    label_list = processor.get_labels()
    output_mode = melbert_main.output_modes["trofi"]
    model = melbert_main.load_pretrained_model(args)

    weights = torch_load_full(model_path, map_location="cpu")
    
    # Load weights into the underlying model, not DataParallel wrapper
    if isinstance(model, torch.nn.DataParallel):
        model.module.load_state_dict(weights)
    else:
        model.load_state_dict(weights)

    model.to(args.device)

    all_guids, eval_loader = melbert_main.load_test_data(
        args, logger, processor, "trofi", label_list, tokenizer, output_mode, fold
    )
    metrics = melbert_main.run_eval(args, logger, model, eval_loader, all_guids, "trofi")
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return {k: float(v) for k, v in metrics.items()}

def run_zero_shot_evaluation():
    expected_vua20 = [f"VUA20_seed{s}" for s in SEEDS]
    missing = [j for j in expected_vua20 if j not in progress["completed_training"]]
    if missing:
        print("Zero-shot evaluation is waiting for VUA-20 training jobs:", missing)
        return

    for seed in SEEDS:
        model_path = Path(progress["completed_training"][f"VUA20_seed{seed}"]["final_model_path"])
        for dataset_name in ["MOH-X", "TroFi"]:
            for fold in range(10):
                key = f"{dataset_name}|seed{seed}|fold{fold}"
                if key in progress["transfer_results"]:
                    continue
                if remaining_session_seconds() <= STOP_RESERVE_MINUTES * 60:
                    write_json_atomic(PROGRESS_PATH, progress)
                    print("Safe stop during zero-shot evaluation; progress persisted.")
                    return
                metrics = evaluate_transfer_fold(model_path, dataset_name, seed, fold)
                progress["transfer_results"][key] = metrics
                write_json_atomic(PROGRESS_PATH, progress)
                print("Saved", key, metrics)

    progress["transfer_completed"] = [
        name for name in ["MOH-X", "TroFi"]
        if all(f"{name}|seed{s}|fold{k}" in progress["transfer_results"] for s in SEEDS for k in range(10))
    ]
    write_json_atomic(PROGRESS_PATH, progress)

run_zero_shot_evaluation()
print("Zero-shot completed:", progress["transfer_completed"])


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***** Running evaluation *****


Evaluating: 100%|██████████| 8/8 [00:00<00:00,  8.67it/s]


  acc = 0.765625
  f1 = 0.8
  precision = 0.7894736842105263
  recall = 0.8108108108108109
Saved MOH-X|seed1|fold0 {'acc': 0.765625, 'precision': 0.7894736842105263, 'recall': 0.8108108108108109, 'f1': 0.8}


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***** Running evaluation *****


... (390 outputs left out)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


***** Running evaluation *****


Evaluating: 100%|██████████| 47/47 [00:05<00:00,  9.32it/s]


  acc = 0.5951742627345844
  f1 = 0.628992628992629
  precision = 0.5565217391304348
  recall = 0.7231638418079096
Saved TroFi|seed5|fold9 {'acc': 0.5951742627345844, 'precision': 0.5565217391304348, 'recall': 0.7231638418079096, 'f1': 0.628992628992629}
Zero-shot completed: ['MOH-X', 'TroFi']


## Tổng hợp kết quả

VUA-18/VUA-20: dùng `best_test_metrics_over_epochs` mà source repo trả về (F1 cao nhất qua 3 lần test). MOH-X/TroFi: trước hết tính trung bình 10 fold cho từng seed, sau đó tính mean và sample standard deviation qua 5 seeds. Notebook xuất CSV/JSON trong `melbert_reproduction/results/`.


In [12]:
def mean_std(rows, keys=("precision", "recall", "f1")):
    out = {}
    for key in keys:
        # sklearn metrics from the source repo are fractions; report as percentage points
        # to match the paper's result tables (e.g. 0.785 -> 78.5).
        vals = [100.0 * float(r[key]) for r in rows if key in r]
        out[f"{key}_mean"] = float(np.mean(vals)) if vals else None
        out[f"{key}_std"] = float(np.std(vals, ddof=1)) if len(vals) > 1 else (0.0 if vals else None)
    return out

summary_rows = []
for dataset_name in ["VUA18", "VUA20"]:
    metrics = []
    for seed in SEEDS:
        entry = progress["completed_training"].get(f"{dataset_name}_seed{seed}")
        if entry:
            metrics.append(entry["best_test_metrics_over_epochs"])
    if metrics:
        summary_rows.append({"dataset": dataset_name, "n_runs": len(metrics), **mean_std(metrics)})

for dataset_name in ["MOH-X", "TroFi"]:
    seed_metrics = []
    for seed in SEEDS:
        fold_metrics = [
            progress["transfer_results"].get(f"{dataset_name}|seed{seed}|fold{k}")
            for k in range(10)
        ]
        fold_metrics = [m for m in fold_metrics if m is not None]
        if fold_metrics:
            seed_metrics.append({
                metric: float(np.mean([m[metric] for m in fold_metrics]))
                for metric in ("precision", "recall", "f1")
            })
    if seed_metrics:
        summary_rows.append({"dataset": dataset_name, "n_runs": len(seed_metrics), **mean_std(seed_metrics)})

import pandas as pd
summary_df = pd.DataFrame(summary_rows)
summary_csv = RESULTS_DIR / "reproduction_summary.csv"
summary_df.to_csv(summary_csv, index=False)
write_json_atomic(RESULTS_DIR / "reproduction_summary.json", summary_rows)

display(summary_df)
print("Saved:", summary_csv)


,dataset,n_runs,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std
0,VUA18,5,80.104618,1.252770,77.328208,1.859206,78.667329,0.368882
1,VUA20,5,75.686804,1.687474,69.758915,1.390703,72.576859,0.234692
2,MOH-X,5,79.240094,0.782332,79.538927,2.706241,79.038379,1.183255
3,TroFi,5,53.275955,0.153084,74.036021,1.578998,61.873505,0.648851


Saved: /kaggle/working/melbert_reproduction/results/reproduction_summary.csv


## Paper reference values for comparison

Các số dưới đây là kết quả MelBERT trong Tables 2, 3 và 6 của paper, không phải output của notebook:

| Dataset | Precision | Recall | F1 |
|---|---:|---:|---:|
| VUA-18 | 80.1 | 76.9 | 78.5 |
| VUA-20 | 76.4 | 68.6 | 72.3 |
| MOH-X (zero-shot from VUA-20) | 79.3 | 79.7 | 79.2 |
| TroFi (zero-shot from VUA-20) | 53.4 | 74.1 | 62.0 |

Khi báo cáo, so sánh mean ± std (đơn vị %) của notebook với các giá trị paper; không kỳ vọng trùng tuyệt đối do GPU/runtime khác và do paper không công bố seed cụ thể trong implementation details.


In [13]:
# Final output inventory. Kaggle saves /kaggle/working contents as notebook output
# when a version is successfully committed/completed.
write_json_atomic(PROGRESS_PATH, progress)
print("Artifacts:")
for path in sorted(ARTIFACT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(ARTIFACT_DIR)}\t{path.stat().st_size/1024/1024:.1f} MB")
print("\nTo continue in a fresh Kaggle session: attach this Notebook Version's output as input and rerun from the top.")


Artifacts:
artifact_manifest.json	0.0 MB
models/VUA18/seed_1/config.json	0.0 MB
models/VUA18/seed_1/merges.txt	0.4 MB
models/VUA18/seed_1/pytorch_model.bin	484.6 MB
models/VUA18/seed_1/training_args.bin	0.0 MB
models/VUA18/seed_1/vocab.json	0.8 MB
models/VUA18/seed_2/config.json	0.0 MB
models/VUA18/seed_2/merges.txt	0.4 MB
models/VUA18/seed_2/pytorch_model.bin	484.6 MB
models/VUA18/seed_2/training_args.bin	0.0 MB
models/VUA18/seed_2/vocab.json	0.8 MB
models/VUA18/seed_3/config.json	0.0 MB
models/VUA18/seed_3/merges.txt	0.4 MB
models/VUA18/seed_3/pytorch_model.bin	484.6 MB
models/VUA18/seed_3/training_args.bin	0.0 MB
models/VUA18/seed_3/vocab.json	0.8 MB
models/VUA18/seed_4/config.json	0.0 MB
models/VUA18/seed_4/merges.txt	0.4 MB
models/VUA18/seed_4/pytorch_model.bin	484.6 MB
models/VUA18/seed_4/training_args.bin	0.0 MB
models/VUA18/seed_4/vocab.json	0.8 MB
models/VUA18/seed_5/config.json	0.0 MB
models/VUA18/seed_5/merges.txt	0.4 MB
models/VUA18/seed_5/pytorch_model.bin	484.6 MB
models/